# exp020_distance_weighted_training_audit train

Audit the exp013 LightGBM no-GR row-distance errors, then compare small distance-weighted LightGBM residual variants under the same GroupKFold-by-well validation.

## Contents

1. Setup and configuration
2. Existing OOF distance audit
3. Distance-weighted training CV
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd

from audit_distance_weighted_training import (
    add_recent_linear,
    audit_existing_oof,
    get_nested,
    load_raw_oof,
    load_yaml,
    run_training_variants,
    train_files,
)
from settings import ExperimentPaths

DEBUG_MAX_WELLS = None
SKIP_TRAINING = False

paths = ExperimentPaths()
config = load_yaml(Path("config.yaml"))
output_dir = paths.artifacts_dir
output_dir.mkdir(parents=True, exist_ok=True)

files = train_files(paths, DEBUG_MAX_WELLS)
variants = get_nested(config, "audit.training_variants.variants", [])

print(f"Experiment: {config['experiment']['name']}")
print(f"Train wells: {len(files)}")
print(f"Raw anchor CV: {get_nested(config, 'audit.raw_clean_cv')}")
print(f"Held-out postprocess reference: {get_nested(config, 'audit.heldout_postprocess_cv')}")
print("Training variants:")
for variant in variants:
    print(f"- {variant['name']}")


## 2. Existing OOF distance audit

Read the exp013 row OOF artifact and compare raw LightGBM, `last_anchor`, `recent_linear`, and the exp014 bucket-shrink parameters by row-distance bucket. This step does not fit a model; it explains where the current anchor is strong or weak.

In [ ]:
oof = load_raw_oof(config)
if DEBUG_MAX_WELLS is not None:
    allowed_wells = {path.name.removesuffix("__horizontal_well.csv") for path in files}
    oof = oof[oof["well_id"].isin(allowed_wells)].reset_index(drop=True)

oof = add_recent_linear(oof, files, config)
candidate_rows, residual_rows, oof_overall = audit_existing_oof(oof, config, output_dir)

candidate_table = pd.DataFrame(candidate_rows)
residual_table = pd.DataFrame(residual_rows)

print("OOF candidate overall RMSE:")
display(candidate_table[candidate_table["segment"] == "overall"].sort_values("rmse"))
print("Raw residual bucket summary:")
display(residual_table)


## 3. Distance-weighted training CV

Fit each candidate only on training-fold wells and score held-out wells. The control reproduces exp013 `lightgbm_no_gr`; the other variants change only the row-distance weighting or split the model into near/mid/far segments.

In [ ]:
training_enabled = bool(get_nested(config, "audit.training_variants.enabled", True))
training_rows: list[dict] = []
importance_rows: list[dict] = []
training_overall: dict[str, float] = {}


In [ ]:
if training_enabled and not SKIP_TRAINING:
    training_rows, importance_rows, training_overall = run_training_variants(
        files,
        config,
        output_dir,
    )
else:
    print("Skipping model training; only the existing OOF audit will be saved.")

if training_rows:
    training_table = pd.DataFrame(training_rows)
    display(training_table[training_table["segment"] == "overall"].sort_values(["fold", "rmse"]))
    pooled = pd.Series(training_overall, name="pooled_rmse").sort_values()
    display(pooled.to_frame())


## 4. Metrics and artifacts

Write the compact summary consumed by `experiment_summary.md`, plus the detailed CSV artifacts used for follow-up analysis.

In [ ]:
best_training_variant = None
if training_overall:
    best_training_variant = min(training_overall, key=training_overall.get)

summary = {
    "experiment": "exp020_distance_weighted_training_audit",
    "status": "completed" if training_overall else "oof_audit_completed",
    "updated_at": datetime.now(UTC).isoformat(),
    "max_wells": DEBUG_MAX_WELLS,
    "raw_anchor_cv": get_nested(config, "audit.raw_clean_cv"),
    "heldout_postprocess_cv": get_nested(config, "audit.heldout_postprocess_cv"),
    "oof_candidate_overall": oof_overall,
    "training_variant_overall": training_overall,
    "best_training_variant": best_training_variant,
    "best_training_cv": training_overall.get(best_training_variant) if best_training_variant else None,
    "artifact_rows": {
        "distance_candidate_metrics": len(candidate_rows),
        "distance_residual_bucket_summary": len(residual_rows),
        "distance_weighted_training_metrics": len(training_rows),
        "distance_weighted_feature_importance": len(importance_rows),
    },
}

(output_dir / "distance_weighted_training_summary.json").write_text(
    json.dumps(summary, indent=2, sort_keys=True) + "\n"
)

metrics = {
    "experiment": "exp020_distance_weighted_training_audit",
    "status": summary["status"],
    "updated_at": summary["updated_at"],
    "cv": summary["best_training_cv"],
    "public_lb": None,
    "raw_anchor_cv": summary["raw_anchor_cv"],
    "heldout_postprocess_cv": summary["heldout_postprocess_cv"],
    "best_training_variant": best_training_variant,
    "training_variant_overall": training_overall,
    "oof_candidate_overall": oof_overall,
}
paths.metrics_path.write_text(json.dumps(metrics, indent=2, sort_keys=True) + "\n")

print(json.dumps(summary, indent=2, sort_keys=True))
